In [ ]:
import time
import csv
import os
import undetected_chromedriver as uc

START_URL = 'https://www.terrabkk.com/freepost/property-list/?d-province_id=1'
OUTPUT_DIR = '/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/terrabkk'
OUTPUT_CSV_FILE = os.path.join(OUTPUT_DIR, 'terrabkk_listing_urls.csv')
PAGE_TIMEOUT = 40
MAX_PAGES = 10

def build_page_url(page):
    if page == 1:
        return 'https://www.terrabkk.com/freepost/property-list/?d-province_id=1'
    else:
        return f'https://www.terrabkk.com/freepost/property-list/{page}?d-province_id=1'

def wait_ready(driver, timeout):
    t0 = time.time()
    while time.time() - t0 < timeout:
        if driver.execute_script("return document.readyState") == "complete":
            return True
        time.sleep(0.2)
    return False

def deep_scroll(driver, rounds=10, pause=0.7):
    h0 = 0
    for _ in range(rounds):
        driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause)
        h = driver.execute_script("return document.body.scrollHeight")
        if h == h0:
            break
        h0 = h

def collect_links_js(driver):
    js = """
    const out = [];
    const items = document.querySelectorAll('.property-item a');
    for (const a of items) {
        const href = a.getAttribute('href') || '';
        if (href.includes('/freepost/show/')) {
            out.push(a.href);
        }
    }
    return Array.from(new Set(out));
    """
    return driver.execute_script(js)

def main():
    options = uc.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    options.add_argument("--disable-blink-features=AutomationControlled")
    driver = uc.Chrome(options=options, version_main=145)

    page = 1
    all_urls = set()
    last_first = ""
    
    driver.get(build_page_url(page))
    wait_ready(driver, PAGE_TIMEOUT)

    while page <= MAX_PAGES:
        print(f"[Page {page}] Scrolling down and collecting links...")
        
        deep_scroll(driver, rounds=15, pause=0.8)
        
        urls_now = collect_links_js(driver)
        
        if not urls_now:
            print(f"  -> No links found")
            break
        
        if page > 1 and urls_now[0] == last_first:
            print(f"  -> Same first link detected, stopping")
            break
        
        last_first = urls_now[0]
        
        for u in urls_now:
            all_urls.add(u)
        
        print(f"  -> Found {len(urls_now)} listings (total: {len(all_urls)})")

        page += 1
        if page > MAX_PAGES:
            break
            
        next_url = build_page_url(page)
        print(f"[Page {page}] Loading next page...")
        driver.get(next_url)
        wait_ready(driver, PAGE_TIMEOUT)
        time.sleep(2)

    driver.quit()

    os.makedirs(OUTPUT_DIR, exist_ok=True)

    with open(OUTPUT_CSV_FILE, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['ListingURL'])
        for u in sorted(all_urls):
            w.writerow([u])
    
    print(f"\n✓ Done. {len(all_urls)} listings saved to {OUTPUT_CSV_FILE}")

if __name__ == "__main__":
    main()

[Page 1] Scrolling down and collecting links...
  -> Found 50 listings (total: 50)
[Page 2] Loading next page...
[Page 2] Scrolling down and collecting links...
  -> Found 50 listings (total: 69)
[Page 3] Loading next page...
[Page 3] Scrolling down and collecting links...
  -> Found 50 listings (total: 88)
[Page 4] Loading next page...
[Page 4] Scrolling down and collecting links...
  -> Found 48 listings (total: 105)
[Page 5] Loading next page...
[Page 5] Scrolling down and collecting links...
  -> Found 51 listings (total: 125)
[Page 6] Loading next page...
[Page 6] Scrolling down and collecting links...
  -> Found 47 listings (total: 141)
[Page 7] Loading next page...
[Page 7] Scrolling down and collecting links...
  -> Found 50 listings (total: 160)
[Page 8] Loading next page...
[Page 8] Scrolling down and collecting links...
  -> Found 51 listings (total: 180)
[Page 9] Loading next page...
[Page 9] Scrolling down and collecting links...
  -> Found 51 listings (total: 200)
[Page 1

In [3]:
import os
import csv
import time
import undetected_chromedriver as uc
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

BASE_DIR = '/home/kongla/Documents/GitHub/Real-estate-Scraping/web-scraping/terrabkk'
INPUT_CSV = os.path.join(BASE_DIR, 'terrabkk_listing_urls.csv')
OUTPUT_CSV = os.path.join(BASE_DIR, 'terrabkk_full_details.csv')

def extract_content(driver):
    try:
        more_btn = driver.find_elements(By.CSS_SELECTOR, 'a.more-detail')
        if more_btn:
            driver.execute_script("arguments[0].click();", more_btn[0])
            time.sleep(0.5)
    except:
        pass

    full_content = []

    try:
        title_element = driver.find_element(By.CSS_SELECTOR, 'h1.freepost-title')
        container = title_element.find_element(By.XPATH, '..')
        full_content.append(container.text.strip())
    except:
        pass

    try:
        col9 = driver.find_element(By.CSS_SELECTOR, '.col-lg-9')
        
        try:
            price = col9.find_element(By.CSS_SELECTOR, '.item-price').text.strip()
            full_content.append(price)
        except:
            pass
            
        try:
            desc = col9.find_element(By.CSS_SELECTOR, '.column-desc').text.strip()
            full_content.append(desc)
        except:
            pass
            
        try:
            addr = col9.find_element(By.CSS_SELECTOR, '.column-address').text.strip()
            full_content.append(addr)
        except:
            pass

        try:
            desc_rows = col9.find_elements(By.CSS_SELECTOR, '.description-row')
            for row in desc_rows:
                text = row.text.strip()
                if text and text not in full_content:
                    full_content.append(text)
        except:
            pass
            
    except:
        pass

    return "\n\n".join(full_content)

def main():
    os.makedirs(BASE_DIR, exist_ok=True)
    
    urls = []
    if os.path.exists(INPUT_CSV):
        with open(INPUT_CSV, 'r', encoding='utf-8') as f:
            reader = csv.reader(f)
            next(reader, None)
            for row in reader:
                if row:
                    urls.append(row[0])
    else:
        urls = [
            "https://www.terrabkk.com/freepost/show/277826/%E0%B8%82%E0%B8%B2%E0%B8%A2-%E0%B8%84%E0%B8%AD%E0%B8%99%E0%B9%82%E0%B8%94-belle-grand-rama-9-%E0%B8%AB%E0%B9%89%E0%B8%AD%E0%B8%87%E0%B8%A3%E0%B8%B5%E0%B9%82%E0%B8%99%E0%B9%80%E0%B8%A7%E0%B8%97%E0%B9%83%E0%B8%AB%E0%B8%A1%E0%B9%88-%E0%B8%9E%E0%B8%A3%E0%B9%89%E0%B8%AD%E0%B8%A1%E0%B8%AD%E0%B8%A2%E0%B8%B9%E0%B9%88-%E2%80%93-%E0%B8%A7%E0%B8%B4%E0%B8%A7%E0%B9%80%E0%B8%A1%E0%B8%B7%E0%B8%AD%E0%B8%87-%E0%B9%80%E0%B8%9F%E0%B8%AD%E0%B8%A3%E0%B9%8C%E0%B8%84%E0%B8%A3%E0%B8%9A",
            "https://www.terrabkk.com/freepost/show/423700/%E0%B8%9A%E0%B9%89%E0%B8%B2%E0%B8%99%E0%B9%80%E0%B8%94%E0%B8%B5%E0%B9%88%E0%B8%A2%E0%B8%A7-2%E0%B8%8A%E0%B8%B1%E0%B9%89%E0%B8%99-72%E0%B8%95%E0%B8%A3%E0%B8%A7-%E0%B8%A5%E0%B8%B2%E0%B8%94%E0%B8%9E%E0%B8%A3%E0%B9%89%E0%B8%B2%E0%B8%A735-%E0%B9%81%E0%B8%A2%E0%B8%816-1"
        ]

    options = uc.ChromeOptions()
    options.add_argument("--no-sandbox")
    options.add_argument("--disable-dev-shm-usage")
    options.add_argument("--disable-gpu")
    options.add_argument("--disable-notifications")
    driver = uc.Chrome(options=options, version_main=145)

    with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as f:
        writer = csv.writer(f)
        writer.writerow(['Post_URL', 'Full_Content'])

        for i, url in enumerate(urls, 1):
            driver.get(url)
            time.sleep(2)
            
            content = extract_content(driver)
            writer.writerow([url, content])

    driver.quit()

if __name__ == "__main__":
    main()